# Python concepts in `parsers.py` - runnable proofs

Companion to `vault/10-concepts/Python Concepts in parsers.md`. Run top to bottom;
each cell *proves* one concept against your real code.

## 1. The mutable-default trap vs `default_factory`

In [ ]:
from dataclasses import dataclass, field

print("(a) modern Python REFUSES mutable dataclass defaults outright:")
try:
    @dataclass
    class Bad:
        items: list = []
except ValueError as e:
    print("   ValueError ->", e)

print()
print("(b) but the hazard is real in plain classes (shared CLASS attribute):")
class PlainBad:
    items = []                     # one list on the class, shared by every instance
a, b = PlainBad(), PlainBad()
a.items.append("polluted")
print("   a.items =", a.items, "| b.items =", b.items, "| identical object?", a.items is b.items)

print()
print("(c) the fix: default_factory calls list() once per instance")
@dataclass
class Good:
    items: list = field(default_factory=list)
g1, g2 = Good(), Good()
g1.items.append("isolated")
print("   g1 =", g1.items, "| g2 =", g2.items, "| identical object?", Good().items is Good().items)

## 2. `slots=True`: memory + typo protection

In [ ]:
import sys
from dataclasses import dataclass, field

@dataclass
class Plain:
    kind: str
    text: str
    rows: list = field(default_factory=list)

@dataclass(slots=True)
class Slotted:
    kind: str
    text: str
    rows: list = field(default_factory=list)

p, s = Plain("paragraph", "x"), Slotted("paragraph", "x")
plain_bytes = sys.getsizeof(p) + sys.getsizeof(p.__dict__)
slot_bytes = sys.getsizeof(s)
print(f"per instance: plain {plain_bytes} B | slots {slot_bytes} B  (saves {plain_bytes-slot_bytes} B)")
print("plain has __dict__:", hasattr(p, "__dict__"), "| slotted:", hasattr(s, "__dict__"))
try:
    s.texxt = "typo"            # silently creates a field on the plain class
    print("plain allows typos: p.texxt =", getattr(p, "texxt", None))
    Plain("a","b").texxt = 1
    print("no error on Plain -> typos become silent bugs")
except AttributeError as e:
    print("slots blocks typos:", e)
try:
    s.texxt = 1
except AttributeError as e:
    print("Slotted raises:", e)

## 3. `Protocol` = structural typing (no inheritance needed)

In [ ]:
from typing import Protocol, runtime_checkable

class DocumentParser(Protocol):
    def parse(self, source: str, title: str): ...

class MarkdownParser:                     # NOTE: inherits nothing
    def parse(self, source: str, title: str):
        return {"title": title, "chars": len(source)}

class NotAParser:
    def other(self): return 1

print("static check (mypy): MarkdownParser satisfies DocumentParser with zero inheritance")
print("parse() call:", MarkdownParser().parse("# Hi", "Doc"))
try:
    print(isinstance(MarkdownParser(), DocumentParser))
except TypeError as e:
    print("isinstance needs @runtime_checkable:", type(e).__name__)

@runtime_checkable
class RuntimeDocumentParser(Protocol):
    def parse(self, source: str, title: str): ...

print("runtime_checkable -> MarkdownParser ok:", isinstance(MarkdownParser(), RuntimeDocumentParser))
print("runtime_checkable -> NotAParser ok  :", isinstance(NotAParser(), RuntimeDocumentParser))

## 4. Registry dispatch + the honest typing wart

In [ ]:
from enum import Enum

class Fmt(str, Enum):
    MD = "md"
    PDF = "pdf"

class MdParser:
    def parse(self, source, title): return f"md:{title}:{len(source)}"

# your file: _PARSERS: dict[DocumentFormat, type]  -> mypy needs `# type: ignore[return-value]`
# correct annotation: type[DocumentParser] (a CALLABLE returning a concrete parser)
_PARSERS: dict[Fmt, type] = {Fmt.MD: MdParser, Fmt.PDF: MdParser}

def get_parser(fmt): return _PARSERS[fmt]()          # instance per call
print(get_parser(Fmt.MD).parse("# x", "Doc"))
print("str,Enum interop:", Fmt.MD == "md", "| json-ish:", Fmt.MD.value)
print("registry keys are values, not types -> singledispatch would NOT fit here")
print("new format = new class + 1 dict line (Open/Closed), no edits to existing parsers")

## 5. Regex: match vs search vs fullmatch

In [ ]:
import re
HEADING = re.compile(r"^(#{1,6})\s+(.*)$")
SEP = r"\|?[\s:\-|]+\|?"

for line in ["# Title", "### Deep", "####### too deep", "#NoSpace", "text # not a heading"]:
    m = HEADING.match(line)
    print(f"{line!r:24s} match -> {m.groups() if m else None}")
print("search() would match mid-line:", bool(re.search(r"#{1,6}", "text # not a heading")))
print("separator rows via fullmatch:", bool(re.fullmatch(SEP, "|---|---|")), bool(re.fullmatch(SEP, "| a | b |")))
print("whitespace collapse:", repr(re.sub(r"\s+", " ", "  a \n\t b   c ").strip()))
print("blank-line split:", re.split(r"\n\s*\n", "p1\n\n\n  \np2"))

## 6. HTMLParser: the event trace behind your blocks

In [ ]:
from html.parser import HTMLParser

class Trace(HTMLParser):
    def handle_starttag(self, tag, attrs): print("  start", tag)
    def handle_endtag(self, tag): print("  end  ", tag)
    def handle_data(self, data):
        if data.strip(): print("  data ", repr(data.strip()))

t = Trace(convert_charrefs=True)
print("callback order (document order, nesting-driven):")
t.feed('<div><h2>Item 7</h2><p>Sales &amp; marketing <ix:nonFraction>1,847</ix:nonFraction></p></div>')
t.close()
print()
print("convert_charrefs=True decoded the entity - there is no '&amp;' in the data above")

## 7. Proof against YOUR real parser

In [ ]:
import sys
sys.path.insert(0, "/Users/pranshu/Desktop/unputdownable/track-a/src")
from app.core.parsers import BlockKind, get_parser
from app.schemas.common import DocumentFormat

MINI = """<div>
<h1>ACME 10-K</h1>
<h2>Item 7 MD&A</h2>
<p>Net sales were <ix:nonFraction>1,847</ix:nonFraction> million.</p>
<h3>Liquidity</h3>
<table><tr><th>Metric</th><th>2023</th></tr><tr><td>Cash</td><td>489</td></tr></table>
<script>var junk = 1;</script>
</div>"""

parsed = get_parser(DocumentFormat.PDF).parse(MINI, "ACME 10-K")
print("blocks:", len(parsed.blocks), "| tables:", parsed.table_count, "| table rows:", parsed.total_table_rows)
for b in parsed.blocks:
    print(f"  {b.kind.value:9s} level={b.level} path={b.section_path} text={b.text[:44]!r}")
print()
print("ix: and script text are absent; the table stayed a TABLE block with rows:")
print(" ", parsed.blocks[-1].table_rows)